# Data

In [1]:
%cd "/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/"

/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python


# Tokenizer

In [2]:
import torch
from utils.Tokenizer.Tokenizer import simpleTokenizer

In [3]:
import pickle

load_path = "/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/data/playlist_dict_prunned.pkl"

with open(load_path, "rb") as f:
    playlist_dict_new = pickle.load(f)

print("Loaded playlists:", len(playlist_dict_new))

Loaded playlists: 1000000


In [4]:
tokenizer = simpleTokenizer(playlist_dict_new)

In [5]:
len(tokenizer.itos)

70229

In [6]:
freq = {}

for ls in playlist_dict_new.values():

    for _ in ls:

        freq[tokenizer.stoi[_]] = freq.get(tokenizer.stoi[_], 0) + 1
    


In [7]:
import numpy as np

vals = np.array(list(freq.values()))

print("tokens       :", len(vals))
print("total_count  :", vals.sum())
print("mean         :", vals.mean())
print("std          :", vals.std())
print("min          :", vals.min())
print("25%          :", np.percentile(vals, 25))
print("median       :", np.median(vals))
print("75%          :", np.percentile(vals, 75))
print("max          :", vals.max())

tokens       : 70229
total_count  : 53521564
mean         : 762.1006137065884
std          : 1905.8007094890263
min          : 100
25%          : 144.0
median       : 240.0
75%          : 547.0
max          : 46574


In [8]:
from gensim.models import Word2Vec
from node2vec import Node2Vec
import networkx as nx
import numpy as np
from collections import defaultdict
from tqdm import tqdm


/Users/rushikesh/python_files/vscode/entity2Vector/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
import pickle

with open("/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/data/test_pids.pkl", "rb") as f:
    test_ids = set(pickle.load(f))

train_playlists = {
    pid: tracks for pid, tracks in playlist_dict_new.items()
    if pid not in test_ids
}

test_playlists = {
    pid: tracks for pid, tracks in playlist_dict_new.items()
    if pid in test_ids
}

In [13]:
# ============================================================
# CELL 2 — Prepare corpus for Word2Vec
# Playlists = sentences, track tokens (as strings) = words
# Gensim expects List[List[str]]
# ============================================================
def build_corpus(playlist_dict, tokenizer):
    """Convert playlist dict to gensim-compatible corpus."""
    corpus = []
    for pid, track_ids in tqdm(playlist_dict.items()):
        # convert track_ids to string tokens (gensim needs strings)
        tokens = [str(tokenizer.stoi[t]) for t in track_ids if t in tokenizer.stoi]
        if len(tokens) > 1:
            corpus.append(tokens)
    return corpus

corpus = build_corpus(train_playlists, tokenizer)
print(f"Total playlists (sentences): {len(corpus)}")
print(f"Sample: {corpus[0][:5]}")

100%|██████████| 950000/950000 [01:38<00:00, 9626.63it/s]  

Total playlists (sentences): 934947
Sample: ['0', '1', '2', '3', '4']


In [16]:
from gensim.models.callbacks import CallbackAny2Vec

class EpochLogger(CallbackAny2Vec):
    def __init__(self):
        self.epoch = 0
    def on_epoch_end(self, model):
        self.epoch += 1
        print(f"Epoch {self.epoch} complete")

w2v_model = Word2Vec(
    sentences=corpus,
    vector_size=128,
    window=5,
    min_count=1,
    sg=1,
    workers=2,
    epochs=5,
    callbacks=[EpochLogger()]
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Epoch 1 complete


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Epoch 2 complete


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Epoch 3 complete


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Epoch 4 complete


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Epoch 5 complete


In [58]:
# w2v_model.save("/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/data/w2v_song_embeddings.model")

# load back
w2v_model = Word2Vec.load("/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/data/w2v_song_embeddings.model")

In [38]:
def get_embedding_w2v(track_token, model):
    if track_token in model.wv:
        return model.wv[track_token]
    return None

# Evaluation W2V

In [42]:
all_tracks = set(tokenizer.stoi.keys())
playlist_tracks = set(t for tracks in train_playlists.values() for t in tracks)

missing = all_tracks - playlist_tracks
print(f"Tracks in tokenizer but not in playlists: {len(missing)}")

Tracks in tokenizer but not in playlists: 0


In [44]:
del all_tracks, playlist_tracks, missing

In [59]:
# First Normalize the vectors
vectors = w2v_model.wv.vectors
w2v_model.wv.vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

In [ ]:
v1 = w2v_model.wv[tokenizer.stoi[test_playlists[549005][0]]]
v2 = w2v_model.wv[tokenizer.stoi[test_playlists[549005][1]]]
v1 @ v2
ct = 0
for i in v1:
    ct += i**2
ct

0.9999999836842335

In [61]:
v1 @ v2

0.55068016

In [64]:
import random

def build_eval_dict(test_playlists, window=5, n_pos=2, n_neg=10, seed=42):
    rng = random.Random(seed)

    # pool for negatives (all tokens seen in test)
    all_tokens = list({tokenizer.stoi[t] for tracks in test_playlists.values() for t in tracks})

    eval_data = {}

    for pid, tracks in test_playlists.items():
        if len(tracks) < window + 2:
            continue

        # pick pivot safely
        i = rng.randint(0, len(tracks) - 1)
        pivot = tokenizer.stoi[tracks[i]]

        # context window (both sides)
        left = max(0, i - window)
        right = min(len(tracks), i + window + 1)
        context = [tokenizer.stoi[t] for idx, t in enumerate(tracks[left:right]) if idx + left != i]

        if len(context) < n_pos:
            continue

        positives = rng.sample(context, n_pos)

        # negatives: not in this playlist context
        neg_pool = [t for t in all_tokens if t not in context and t != pivot]
        negatives = rng.sample(neg_pool, n_neg)

        eval_data[pid] = {
            "pivot": pivot,
            "positives": positives,
            "negatives": negatives,
        }

    return eval_data

In [65]:
eval_dict = build_eval_dict(test_playlists, window=5)
print(len(eval_dict))

46929


In [62]:
import numpy as np

def evaluate_embeddings(X, eval_dict, k_recall=2):
    recalls = []
    ranks = []

    for ex in tqdm(eval_dict.values()):
        pivot = ex["pivot"]
        positives = ex["positives"]
        negatives = ex["negatives"]

        candidates = positives + negatives
        
        pivot_vec = X[str(int(pivot))]
        
        sims = [pivot_vec @ X[str(int(t))] for t in candidates]

        # rank (descending similarity)
        order = np.argsort(sims)[::-1]

        # positions of positives
        pos_ranks = []
        for idx in range(len(positives)):
            rank = np.where(order == idx)[0][0] + 1  # 1-based
            pos_ranks.append(rank)

        # Recall@k
        recall = sum(r <= k_recall for r in pos_ranks) / len(pos_ranks)
        recalls.append(recall)

        # Average rank
        ranks.extend(pos_ranks)

    return {
        "recall@{}".format(k_recall): np.mean(recalls),
        "avg_rank": np.mean(ranks),
    }

In [63]:
metrics = evaluate_embeddings(w2v_model.wv, eval_dict, k_recall=2)
print(metrics)

100%|██████████| 46929/46929 [00:00<00:00, 61610.40it/s]

{'recall@2': 0.8503590530375674, 'avg_rank': 1.777749366063628}


# Node 2 Vec

In [10]:
from scipy.sparse import load_npz

cooc = load_npz("/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/data/cooc_matrix.npz")
cooc[0,1001], cooc[1001,0]


(2.0, 2.0)

In [11]:

print(f"Filled : {100*cooc.nnz/(cooc.shape[0] * cooc.shape[1]):.2f}%")


Filled : 2.93%


In [12]:

# Postitive pointwise mutual information
import numpy as np
from scipy.sparse import csr_matrix

def csr_to_ppmi(cooc: csr_matrix):
    cooc = cooc.tocsr().astype(np.float64)

    # totals
    total = cooc.sum()
    row_sum = np.array(cooc.sum(axis=1)).flatten()
    col_sum = np.array(cooc.sum(axis=0)).flatten()

    # avoid divide-by-zero
    row_sum[row_sum == 0] = 1
    col_sum[col_sum == 0] = 1

    # iterate over nonzeros only
    rows, cols = cooc.nonzero()
    data = cooc.data

    # PMI
    pmi = np.log((data * total) / (row_sum[rows] * col_sum[cols]))

    # PPMI
    pmi[pmi < 0] = 0

    return csr_matrix((pmi, (rows, cols)), shape=cooc.shape)

ppmi = csr_to_ppmi(cooc)

In [ ]:
import networkx as nx
from scipy.sparse import find
from tqdm import tqdm

def ppmi_to_graph(ppmi):
    G = nx.Graph()
    rows, cols, weights = find(ppmi)
    for u, v, w in tqdm(zip(rows, cols, weights), total=len(weights), desc="Building graph"):
        if u < v:  # avoid duplicate edges since undirected
            G.add_edge(str(u), str(v), weight=w)
    return G

G = ppmi_to_graph(ppmi)
print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

Building graph:  53%|█████▎    | 66427885/125842415 [14:44<13:48, 71674.66it/s]  

In [ ]:

# ============================================================
# CELL 4 — Build co-occurrence graph for node2vec
# Edge weight = number of times two tracks co-occur within window
# ============================================================
def build_cooccurrence_graph(corpus, window=10):
    G = nx.Graph()
    edge_weights = defaultdict(int)
    
    for playlist in tqdm(corpus, desc="Building graph"):
        for i, node in enumerate(playlist):
            ctx = playlist[max(0, i-window): i] + playlist[i+1: i+window+1]
            for neighbor in ctx:
                if node != neighbor:
                    key = tuple(sorted([node, neighbor]))
                    edge_weights[key] += 1
    
    for (u, v), w in edge_weights.items():
        G.add_edge(u, v, weight=w)
    
    return G

G = build_cooccurrence_graph(corpus, window=10)
print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")


# ============================================================
# CELL 5 — Train node2vec
# ============================================================
n2v = Node2Vec(
    G,
    dimensions=128,
    walk_length=30,
    num_walks=10,
    p=1,          # return parameter
    q=1,          # in-out parameter (q>1 = BFS-like, q<1 = DFS-like)
    workers=4
)

n2v_model = n2v.fit(
    window=10,
    min_count=1,
    sg=1,
    epochs=10
)

print(f"node2vec vocab size: {len(n2v_model.wv)}")


# ============================================================
# CELL 6 — Helper: get embedding by track_id
# ============================================================


def get_embedding_n2v(track_id, model, tokenizer):
    token = str(tokenizer.stoi.get(track_id))
    if token in model.wv:
        return model.wv[token]
    return None


# ============================================================
# CELL 7 — Quick sanity check: similar songs
# ============================================================
def similar_tracks(track_id, model, tokenizer, track_name_map, topn=5):
    token = str(tokenizer.stoi.get(track_id))
    similar = model.wv.most_similar(token, topn=topn)
    results = []
    for tok, score in similar:
        tid = tokenizer.itos[int(tok)]
        name = track_name_map.get(tid, tid)
        results.append((name, round(score, 4)))
    return results

# Example usage:
# similar_tracks("some_track_id", w2v_model, tokenizer, track_name_map)
# similar_tracks("some_track_id", n2v_model, tokenizer, track_name_map)